<a href="https://colab.research.google.com/github/ZanebRA/urdu-ocr-codesaviours-si26-zaneb/blob/main/SI26_Week4_Zaneb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU Available: True
GPU: Tesla T4


In [1]:
!pip install -q sentencepiece tiktoken

In [2]:
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

In [4]:
import transformers
print(transformers.__version__)

5.13.1


In [5]:
!pip show tokenizers

Name: tokenizers
Version: 0.22.2
Summary: 
Home-page: https://github.com/huggingface/tokenizers
Author: 
Author-email: Nicolas Patry <patry.nicolas@protonmail.com>, Anthony Moi <anthony@huggingface.co>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: huggingface-hub
Required-by: torchtune, transformers


In [6]:
!pip uninstall -y transformers tokenizers
!pip install -q transformers==4.41.2 tokenizers==0.19.1 sentencepiece tiktoken

Found existing installation: transformers 5.13.1
Uninstalling transformers-5.13.1:
  Successfully uninstalled transformers-5.13.1
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [1]:
import transformers
import tokenizers

print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Transformers: 4.41.2
Tokenizers: 0.19.1


In [3]:
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")

model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-printed")

model.to(device)

print("✅ Model loaded successfully!")
print("Using device:", device)

Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-base-printed and are newly initialized: ['encoder.pooler.dense.bias', 'encoder.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model loaded successfully!
Using device: cuda


In [4]:
# Configure model for training

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.sep_token_id

model.config.vocab_size = model.config.decoder.vocab_size

print("✅ Model configuration completed!")

✅ Model configuration completed!


In [5]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=5e-5
)

print("✅ Optimizer created successfully!")

✅ Optimizer created successfully!


In [6]:
# Put model into training mode
model.train()

print("✅ Model is ready for training!")

✅ Model is ready for training!


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [25]:
!pip install -q transformers torch pillow pandas
import torch
from torch.utils.data import Dataset
from transformers import TrOCRProcessor
from PIL import Image
import pandas as pd
class UrduOCRDataset(Dataset):

    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        print(f"Dataset loaded: {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        # Image path
        image_path = "/content/drive/MyDrive/data/" + row["image"]

        # Load image
        image = Image.open(image_path).convert("RGB")

        # Process image
        encoding = self.processor(image, return_tensors="pt")
        pixel_values = encoding.pixel_values.squeeze()

        # Process text
        labels = self.processor.tokenizer(
    row["text"],
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
).input_ids.squeeze(0)

        labels = torch.tensor(labels)

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

print("✅ Dataset class created successfully!")

✅ Dataset class created successfully!


In [26]:
from transformers import TrOCRProcessor

processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-printed",
    use_fast=False
)

dataset = UrduOCRDataset(
    "/content/drive/MyDrive/data/data/labels.csv",
    processor
)

print("Total samples:", len(dataset))

sample = dataset[0]

print("Pixel Values Shape:", sample["pixel_values"].shape)
print("Labels Shape:", sample["labels"].shape)

Dataset loaded: 200 samples
Total samples: 200
Pixel Values Shape: torch.Size([3, 384, 384])
Labels Shape: torch.Size([128])


/tmp/ipykernel_4651/4087496317.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels)


In [27]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size]
)

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))

Training samples: 160
Testing samples: 40


In [28]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False
)

print("Train DataLoader created successfully!")
print("Test DataLoader created successfully!")

Train DataLoader created successfully!
Test DataLoader created successfully!


In [29]:
batch = next(iter(train_loader))

print("Pixel Values Shape:", batch["pixel_values"].shape)
print("Labels Shape:", batch["labels"].shape)

/tmp/ipykernel_4651/4087496317.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels)


Pixel Values Shape: torch.Size([4, 3, 384, 384])
Labels Shape: torch.Size([4, 128])


In [30]:
from tqdm import tqdm

num_epochs = 3

model.train()

for epoch in range(num_epochs):

    total_loss = 0

    progress_bar = tqdm(train_loader)

    for batch in progress_bar:

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            pixel_values=pixel_values,
            labels=labels
        )

        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        progress_bar.set_description(
            f"Epoch {epoch+1}/{num_epochs}"
        )
        progress_bar.set_postfix(
            loss=loss.item()
        )

    avg_loss = total_loss / len(train_loader)

    print(f"\nEpoch {epoch+1} Average Loss: {avg_loss:.4f}")

  0%|          | 0/40 [00:00<?, ?it/s]/tmp/ipykernel_4651/4087496317.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels)
Epoch 1/3: 100%|██████████| 40/40 [02:34<00:00,  3.87s/it, loss=3.05]



Epoch 1 Average Loss: 2.8361


Epoch 2/3: 100%|██████████| 40/40 [00:44<00:00,  1.12s/it, loss=2.26]



Epoch 2 Average Loss: 2.1942


Epoch 3/3: 100%|██████████| 40/40 [00:43<00:00,  1.09s/it, loss=1.56]


Epoch 3 Average Loss: 1.8865


In [31]:
from tqdm import tqdm

model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch in tqdm(test_loader):
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        generated_ids = model.generate(pixel_values)

        pred_texts = processor.batch_decode(generated_ids, skip_special_tokens=True)
        label_texts = processor.batch_decode(labels, skip_special_tokens=True)

        for pred, true in zip(pred_texts, label_texts):
            if pred.strip() == true.strip():
                correct += 1
            total += 1

accuracy = (correct / total) * 100
print(f"Accuracy: {accuracy:.2f}%")

  0%|          | 0/10 [00:00<?, ?it/s]/tmp/ipykernel_4651/4087496317.py:40: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = torch.tensor(labels)
/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1168: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
100%|██████████| 10/10 [00:52<00:00,  5.22s/it]

Accuracy: 0.00%


In [33]:
model.save_pretrained("trocr_urdu_model")
processor.save_pretrained("trocr_urdu_model")

print("Model saved successfully!")

Model saved successfully!


In [34]:
import shutil

shutil.copytree(
    "trocr_urdu_model",
    "/content/drive/MyDrive/trocr_urdu_model",
    dirs_exist_ok=True
)

print("Model copied to Google Drive!")

Model copied to Google Drive!
